## 1 · Principal Activity Report — column names

In [6]:
import os, pandas as pd
pd.set_option('display.max_columns', 40, 'display.width', 200)

DATA = next(p for p in ['../../data/ARES_2023_68A','../data/ARES_2023_68A',
                        'data/ARES_2023_68A'] if os.path.isdir(p))
PAR = pd.read_excel(os.path.join(DATA, 'principal_activity_report.xlsx'), header=0)
print('columns:', list(PAR.columns))
print('rows:', len(PAR))
PAR.head(6)

columns: ['issuer_name', 'facility', 'loanx_id_cusip', 'transaction_amount', 'transaction_date', 'transaction_type']
rows: 630


,issuer_name,facility,loanx_id_cusip,transaction_amount,transaction_date,transaction_type
0,NaN,Beginning Balance,NaN,11660136.83,2026-04-17,NaN
1,"VS Buyer, LLC",Cov Lite TLB,LX266922,7.46,2026-04-17,Facility Purchase - Net Economic Benefit
2,Pregis Topco LLC,Term Loan (05/25),LX263493,0.35,2026-04-17,Facility Sale - Net Economic Benefit
3,"CPV Fairview, LLC",Term Loan,LX271494,1.88,2026-04-17,Facility Purchase - Net Economic Benefit
4,Kaman Corporation,Term Loan B (01/25),LX258138,5350.52,2026-04-17,Facility Purchase - Permanent Reduction
5,Kaman Corporation,Term Loan B (01/25),LX258138,8025.77,2026-04-17,Facility Purchase - Permanent Reduction


The file has **no** dedicated *scheduled* / *unscheduled* principal columns.
The only classification of a flow is the free-text `transaction_type`

## 2 · Every `transaction_type` value

In [7]:
print(PAR['transaction_type'].value_counts(dropna=False).to_string())

transaction_type
Loan - Interest Payment                                                                                              263
Facility - Purchase                                                                                                   49
Facility - Sale                                                                                                       47
Facility Sale - Delayed Compensation Fee Paid                                                                         44
Facility Sale - Cost of Carry Fee Received                                                                            43
Facility Purchase - Permanent Reduction                                                                               35
Facility Purchase - Delayed Compensation Fee Received                                                                 34
Facility Purchase - Cost of Carry Fee Paid                                                                            34
Facility - Payd

**Finding.** The report is a *transaction ledger* of purchases, sales, interest,
fees, and one lumped **`Facility - Paydown`** category. There is:
- **no `Scheduled Principal`** (amortization) field or type, and
- **no `Unscheduled Principal` / `Prepayment`** field or type.

`Facility - Paydown` mixes any principal repayment together and does **not** flag
whether a given paydown was contractual amortization or a voluntary prepayment.
There is also no defaults/recoveries column to net out.


**A true realized CPR cannot be computed from these inputs.**

The CPR method needs, per period:
`periodic_rate = unscheduled_principal / (beginning_balance − scheduled_principal)`.

- ✅ `beginning_balance`: available from the CLP `Balance` row.
- ❌ **`scheduled_principal`**: not reported.
- ❌ **`unscheduled_principal` (prepayments)**: not reported.


## 3 ESTIMATE CPR

The true quarterly CPR is not computable (section 5). As an **estimate only**, we
proxy unscheduled principal with the **`Facility - Paydown`** rows 

**Explicit assumptions (this is an estimate, not a labeled split):**
- `unscheduled ≈ Σ Facility - Paydown`. **Excludes** `Facility Purchase
- `scheduled ≈ 0` (institutional TLBs amortize ~1%/yr ≈ negligible per period).

In [ ]:
# single-period prepayment ESTIMATE
d = pd.to_datetime(PAR['transaction_date'], errors='coerce')
start_month = d.min().to_period('M')          # 2026-04
period_label = f"{d.min().date()} -> {d.max().date()}"

# unscheduled proxy = Facility - Paydown only (exclude purchase-side 'Permanent Reduction')
unscheduled = PAR.loc[PAR['transaction_type'] == 'Facility - Paydown', 'transaction_amount'].sum()
scheduled = 0.0                                # assumed ~0 (near-bullet TLBs)

# beginning balance from CLP: 'Balance (M)' row (thousands -> dollars), period start month
hdr = [str(x).strip() for x in clp.iloc[0, :]]
col = hdr.index(start_month.strftime('%m/%Y'))
beginning_balance = float(clp.iloc[1, col]) * 1_000   # Balance (M) is actually in thousands

periodic_rate = unscheduled / (beginning_balance - scheduled)
cpr_q = (1 - (1 - periodic_rate) ** 4) * 100    # task's quarterly framing (n=4)
cpr_m = (1 - (1 - periodic_rate) ** 12) * 100   # monthly basis (period is ~1 month)

tbl = pd.DataFrame([{
    'period': period_label,
    'beginning_balance': round(beginning_balance, 0),
    'scheduled': round(scheduled, 0),
    'unscheduled': round(unscheduled, 0),
    'periodic_rate': round(periodic_rate, 5),
    'CPR_est_%': round(cpr_m, 2),
}])
print("SINGLE-PERIOD ESTIMATE (n=1, proxy — NOT a measured quarterly CPR)\n")
print(tbl.to_string(index=False))
print(f"\nperiodic prepay rate: {periodic_rate*100:.3f}% over the period")
print(f"annualized CPR — monthly basis (^12): {cpr_m:.2f}%   | quarterly basis (^4): {cpr_q:.2f}%")
print("Annualization basis is uncertain from one sub-period; the ^12 figure assumes "
      "this is a monthly collection period.")

SINGLE-PERIOD ESTIMATE (n=1, proxy — NOT a measured quarterly CPR)

                  period  beginning_balance  scheduled  unscheduled  periodic_rate  CPR_est_%
2026-04-17 -> 2026-05-05        488841000.0        0.0    3908086.0        0.00799       9.18

periodic prepay rate: 0.799% over the period
annualized CPR — monthly basis (^12): 9.18%   | quarterly basis (^4): 3.16%
Annualization basis is uncertain from one sub-period; the ^12 figure assumes this is a monthly collection period.


In [ ]:
#  write the output table to CSV 
out_tbl = tbl.copy()
out_tbl['CPR_est_%_monthly_basis']   = round(cpr_m, 2)
out_tbl['CPR_est_%_quarterly_basis'] = round(cpr_q, 2)
out_tbl['unscheduled_source'] = 'Facility - Paydown (proxy)'
out_tbl['scheduled_assumption'] = 'assumed 0 (near-bullet TLBs)'
out_tbl['note'] = 'ESTIMATE, n=1 single collection period — not a measured quarterly CPR'
out_tbl = out_tbl.drop(columns=['CPR_est_%'])

OUT = os.path.join('output', 'cpr_estimate.csv')
os.makedirs('output', exist_ok=True)
out_tbl.to_csv(OUT, index=False)
print('wrote', os.path.abspath(OUT))
print(out_tbl.T.to_string())

wrote /Users/amine/Documents/Columbia/Classes/ENGIE 4700 - Summer Project/correlation_and_tail_risk_in_clo_tranches/project/notebooks/output/cpr_estimate.csv
                                                                                               0
period                                                                  2026-04-17 -> 2026-05-05
beginning_balance                                                                    488841000.0
scheduled                                                                                    0.0
unscheduled                                                                            3908086.0
periodic_rate                                                                            0.00799
CPR_est_%_monthly_basis                                                                     9.18
CPR_est_%_quarterly_basis                                                                   3.16
unscheduled_source                                                